# Applications of Machine Learning - Assignment 3

**Submitted by:<br>
Urvi Sharma <br>
Roll No: AM.EN.P2ARI2022 <br>
Date : 02-03-2020**

# Recommendation Systems

## (Collaborative Filtering)

A Recommendation System aims at suggesting optimal recommendation of items to
various users. 

Collaborative Filtering approach is based on the ratings of the users for
various items in a domain.

![alt text](rs.png "Title")

**1. Give a formal description of this application in terms of Task, Experience, and
Performance.**

**(Make it a well posed problem)**

Task: Which products can be suggested to a person buying a particular product?

Experience: Rating Data (Utility Matrix)

Performance: What percentage of the recommended products were purchased?

**2. A dataset is provided consisting of the ratings of various products from Amazon users.** 

**Write a function Utility_matrix(data) that converts the uploaded dataset ‘amazon_rating’ into an Utility matrix where columns represent items and rows represent users and the values represent rating.**

**For instance, consider the following matrix**

![alt text](t1.png "Title")

**The given matrix should be converted to Utility matrix as follows.** 

**Item 1 corresponds to Item 9707716436 and Item 2 corresponds to 9707716371. Similarly, for users.**

![alt text](t2.png "Title")

In [1]:
import pandas as pd

In [2]:
amazon_rating_data = pd.read_csv('rating-user.csv')

In [3]:
amazon_rating_data.head(5)

,Item,User,Rating
0,B000K2PJ4K,ALJ66O1Y6SLHA,5
1,B000K2PJ4K,ALJ66O1Y6SLHA,5
2,B000K2PJ4K,ALJ66O1Y6SLHA,5
3,B000K2PJ4K,ALJ66O1Y6SLHA,5
4,B000K2PJ4K,ALJ66O1Y6SLHA,5


In [4]:
number_of_items = amazon_rating_data['Item'].value_counts().count() 
number_of_items

31

In [5]:
number_of_users = amazon_rating_data['User'].value_counts().count()
number_of_users

406

Rows (Users): 2472

Columns (Items) = 37

In [6]:
amazon_rating_data['Rating'].value_counts()

5    2158
4     471
3     337
1     117
2      93
Name: Rating, dtype: int64

In [7]:
items = {}
users = {}

In [8]:
df_items = pd.DataFrame(amazon_rating_data['Item'].value_counts()).reset_index()

In [9]:
item_counter = 1
for item in list(df_items.iloc[:,0]):
    items[item] = f"Item {item_counter}"
    item_counter = item_counter + 1

In [10]:
df_users = pd.DataFrame(amazon_rating_data['User'].value_counts()).reset_index()

In [11]:
user_counter = 1
for user in list(df_users.iloc[:,0]):
    users[user] = f"User {user_counter}"
    user_counter = user_counter + 1

In [12]:
def convert_user(original_user):
    if original_user in users:
        return users[original_user]
    
def convert_item(original_item):
    if original_item in items:
        return items[original_item]

In [13]:
amazon_rating_data_x = amazon_rating_data.copy()
amazon_rating_data_x['User'] = amazon_rating_data_x['User'].apply(convert_user)
amazon_rating_data_x['Item'] = amazon_rating_data_x['Item'].apply(convert_item)

In [14]:
amazon_rating_data_x.head()

,Item,User,Rating
0,Item 13,User 388,5
1,Item 13,User 388,5
2,Item 13,User 388,5
3,Item 13,User 388,5
4,Item 13,User 388,5


In [15]:
import numpy as np

In [16]:
data = np.full([number_of_users, number_of_items], None)

In [17]:
data.shape

(406, 31)

In [18]:
rownames = list(df_users.iloc[:,0])
colnames = list(df_items.iloc[:,0])
utility_matrix = pd.DataFrame(data, index=users.values(), columns=items.values())

In [19]:
utility_matrix.head(1)

,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,Item 10,...,Item 22,Item 23,Item 24,Item 25,Item 26,Item 27,Item 28,Item 29,Item 30,Item 31
User 1,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [20]:
for idx, row in amazon_rating_data_x.iterrows():
    utility_matrix.loc[row.loc['User']][row.loc['Item']] = row.loc['Rating']

**3. Write a function Normalize(U) to normalize the ratings of the users for the items.**

In [21]:
def normalize(u):
    u_normalized = u.copy()
    for idx, row in u.iterrows():
        row = [0  if v is None else v for v in row]
        avg = sum(row)/np.count_nonzero(row)
        u_normalized.loc[idx] = [(v - avg)  if v is not None else v for v in u_normalized.loc[idx]]
        #print(u_normalized.loc[idx])
    return u_normalized

In [22]:
normalize(utility_matrix.head(5))

,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,Item 10,...,Item 22,Item 23,Item 24,Item 25,Item 26,Item 27,Item 28,Item 29,Item 30,Item 31
User 1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None,...,None,None,None,None,None,None,None,None,None,None
User 2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None,...,None,None,None,None,None,None,None,None,None,None
User 3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None,...,None,None,None,None,None,None,None,None,None,None
User 4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None,...,None,None,None,None,None,None,None,None,None,None
User 5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None,...,None,None,None,None,None,None,None,None,None,None


**4. Write a function PearsonCorr(x,y) to find the similarity between the ratings of the items rated by both users x and y.**

![alt text](eq1.png "Title")

In [23]:
from scipy.stats import pearsonr

In [24]:
def pearson_correlation_coefficient(X, Y):
    X = np.array([0  if x is None else x for x in X])
    Y = np.array([0  if y is None else y for y in Y])
    #corr, _ = pearsonr(X, Y)
    n = np.dot((X - X.mean()), (Y - Y.mean()))
    d = np.sqrt(np.sum((X - X.mean()) ** 2)) * np.sqrt(np.sum((Y - Y.mean()) ** 2))
    return n / d
    #return corr

In [25]:
A = np.array([4, None, None, 5, 1, None, None])
B = np.array([5, 5, 4, None, None, None, None])
C = np.array([None, None, None, 2, 4, 5, None])

In [26]:
pearson_correlation_coefficient(A, A)

0.9999999999999999

In [27]:
pearson_correlation_coefficient(A, B)

0.0

In [28]:
pearson_correlation_coefficient(A, C)

-0.06185567010309282

**5. Write a function NearestNbrs(U,q,k) that takes the normalized utility matrix ‘U’,the query ‘q’ as the rating vector of a user, and finds the best ‘k’ neighbours from ‘U’ based on the similarity metric Pearson Correlation coefficient.**

In [29]:
def nearest_neighbours(U, Q, k):
    log = {}
    Q = np.array([0  if q is None else q for q in Q])
    for idx, R in U.iterrows():
        R = [0  if r is None else r for r in R]
        #s, _ = pearsonr(R, Q)
        s = pearson_correlation_coefficient(R, Q)
        log[idx] = s
        
    return sorted(log, key=log.get, reverse=True)[1:k+1]

In [30]:
Test_U = pd.DataFrame([[1, None, 3, None, None, 5, None, None, 5, None, 4, None],
                       [None, None, 5, 4, None, None, 4, None, None, 2, 1, 3],
                       [2, 4, None, 1, 2, None, 3, None, 4, 3, 5, None],
                       [None, 2, 4, None, 5, None, None, 4, None, None, 2, None],
                       [None, None, 4, 3, 4, 2, None, None, None, None, 2, 5],
                       [1, None, 3, None, 3, None, None, 2, None, None, 4, None]
                      ])

In [31]:
Test_U = Test_U.replace({np.nan: None})

In [32]:
Test_Q = np.array([1, None, 3, None, None, 5, None, None, 5, None, 4, None])

In [33]:
nearest_neighbours(normalize(Test_U), Test_Q, 2)

[0, 5]

**6. Write a function PredictRating(x,s) that predicts the rating of user x for item i based on the following formula:**

![alt text](eq2.png "Title")

In [34]:
def predict_rating(U, X, i, n):
    nns = nearest_neighbours(normalize(U), X, n)
    X = np.array([0  if x is None else x for x in X])
    sum_s=0
    n=0
    for idx, R in U.iterrows():
        if idx in nns:
            R = np.array([0  if r is None else r for r in R])
            n = n + np.dot(pearson_correlation_coefficient(X, R), R[i])
            sum_s = sum_s + pearson_correlation_coefficient(X, R)
    return n/sum_s 

In [35]:
#change no of neighbours 2,4,6,8

In [36]:
predict_rating(Test_U, Test_Q, 4, 2)

0.4705414738406447

## Viva Assignment

**for 2 neighbours**

In [37]:
normalize(Test_U)

,0,1,2,3,4,5,6,7,8,9,10,11
0,-2.6,None,-0.6,None,None,1.4,None,None,1.4,None,0.400000,None
1,None,None,1.833333,0.833333,None,None,0.833333,None,None,-1.166667,-2.166667,-0.166667
2,-1.0,1.0,None,-2.0,-1.0,None,0.0,None,1.0,0.0,2.000000,None
3,None,-1.4,0.6,None,1.6,None,None,0.6,None,None,-1.400000,None
4,None,None,0.666667,-0.333333,0.666667,-1.333333,None,None,None,None,-1.333333,1.666667
5,-1.6,None,0.4,None,0.4,None,None,-0.6,None,None,1.400000,None


In [38]:
Test_Q

array([1, None, 3, None, None, 5, None, None, 5, None, 4, None],
      dtype=object)

In [39]:
nearest_neighbours(normalize(Test_U), Test_Q, 4)

[0, 5, 1, 3]

In [40]:
predict_rating(Test_U, Test_Q, 4, 2)

0.4705414738406447

**for 4 neighbours**

In [41]:
nearest_neighbours(normalize(Test_U), Test_Q, 4)

[0, 5, 1, 3]

In [42]:
predict_rating(Test_U, Test_Q, 4, 4)

-0.06590926334929276

**for 6 neighbours**

In [43]:
nearest_neighbours(normalize(Test_U), Test_Q, 6)

[0, 5, 1, 3, 4]

In [44]:
predict_rating(Test_U, Test_Q, 4, 6)

-0.0659092633492929

**for 8 neighbours**

In [45]:
nearest_neighbours(normalize(Test_U), Test_Q, 8)

[0, 5, 1, 3, 4]

In [46]:
predict_rating(Test_U, Test_Q, 4, 8)

-0.0659092633492929